<a href="https://colab.research.google.com/github/Kinds-of-Intelligence-CFI/measurement-layouts/blob/main/analysis/measurement-layouts/object_permanence_measurement_layout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparative Object Permanence Measurement Layouts

Authors: K. Voudouris

## INIT

In [ ]:
!pip install pymc --quiet
!pip install numpy --quiet
!pip install arviz --quiet
!pip install jax --quiet
!pip install numpyro --quiet

In [ ]:
%env XLA_PYTHON_CLIENT_PREALLOCATE = false
%env XLA_PYTHON_CLIENT_ALLOCATOR=platform

In [ ]:
import arviz as az
import cloudpickle
import gc
import graphviz
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import pymc as pm
import random as rm
import seaborn as sns
import jax
import numpyro
import pymc.sampling.jax as pmjax
numpyro.set_platform("gpu")

from collections import defaultdict
from google.colab import userdata
from IPython.display import Image
from scipy import stats
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from google.colab import files
from pymc import model

print(f"Running on PyMC v{pm.__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Stored secrets for folders
training_data_model_folder = userdata.get('train_folder')
full_data_model_folder = userdata.get('full_folder')
figures_folder = userdata.get('fig_folder')
csv_folder = userdata.get('csv_folder')

## Load Data


In [ ]:
long_data_url = 'https://raw.githubusercontent.com/Kinds-of-Intelligence-CFI/measurement-layouts/refs/heads/main/object_permanence_voudouris2024/results_final_clean_long.csv'
long_data = pd.read_csv(long_data_url)

## Define Some Useful Functions

In [ ]:
def logistic_general(x, min, max, c = 0, p = 0.99):

  """
  Generalized version of the logistic where min can be any number, not just 0 as above.

  :param min: The min ability/demand
  :param max: The max ability/demand
  :param c: Set different from 0 when we are dealing with multiple choice responses; e.g. c is 0.5 when the response is yes/no, and c is 0.25 if there are 4 possible choices
  :param p: The probability we want to set the max margin to (e.g. 0.99 or 0.999); the min margin will be set to 1 - p
  :param k: The slope of the logistic
  """
  max_n = max - min
  k = - np.log((1 - p )/(p - c)) / max_n
  return c + ((1 - c) / (1 + np.exp(-k * x)))

# A function for pulling out the means and standard deviations for abilities of interest
def analyzeAgentResults(agentData, abilitiesToShow):

  abilityMeans = {}
  abilitySDs = {}

  for a in abilitiesToShow: #iterate through each ability, add posterior mean to dataframe, and plot posterior

    posteriorMean = float(np.mean(agentData['posterior'][a])) # calculate posterior mean
    posteriorSD = float(np.std(agentData['posterior'][a])) #calculate posterior sd
    abilityMeans[a] = posteriorMean
    abilitySDs[a] = posteriorSD

  return abilityMeans, abilitySDs

def predict(m, trace, relevantData):
  with m:
    predictions = pm.sample_posterior_predictive(trace, var_names=["choiceP", "successP"], return_inferencedata=False, predictions=True, extend_inferencedata=False)
    predictionSuccessChainRuns = predictions["successP"][:,:,0:len(relevantData)]
    predictionsSuccessInstance = np.mean(predictionSuccessChainRuns, (0,1))
    predictionChoiceChainRuns = predictions["choiceP"][:,:,0:len(relevantData)]
    predictionsChoiceInstance = np.mean(predictionChoiceChainRuns, (0,1))

    successes = relevantData['success'].to_numpy()
    choices = relevantData['correctChoice'].to_numpy()

    return predictionsSuccessInstance, predictionsChoiceInstance, successes, choices

def brierScore(preds, outs):
    return 1/len(preds) * sum( (preds-outs)**2 )

def brierDecomp(preds, outs):

  brier= 1/len(preds) * sum( (preds-outs)**2 )
  ## bin predictions
  bins = np.linspace(0,1,11)
  binCenters = (bins[:-1] +bins[1:]) /2
  binPredInds = np.digitize(preds,binCenters)
  binnedPreds = bins[binPredInds]

  binTrueFreqs = np.zeros(10)
  binPredFreqs = np.zeros(10)
  binCounts = np.zeros(10)

  for i in range(10):
      idx = (preds >= bins[i]) & (preds < bins[i+1])

      binTrueFreqs[i] = np.sum(outs[idx])/np.sum(idx) if np.sum(idx) > 0 else 0
      binPredFreqs[i] = np.mean(preds[idx]) if np.sum(idx) > 0 else 0
      binCounts[i] = np.sum(idx)

  calibration = np.sum(binCounts * (binTrueFreqs - binPredFreqs) ** 2) / np.sum(binCounts) if np.sum(binCounts) > 0 else 0
  refinement = np.sum(binCounts * (binTrueFreqs *(1 - binTrueFreqs))) / np.sum(binCounts) if np.sum(binCounts) > 0 else 0
  return brier, calibration, refinement

# Pickle objects
def save_object(obj, filename):
    with open(filename, 'wb') as outp:  # Overwrites any existing file.
        cloudpickle.dump(obj, outp, pickle.HIGHEST_PROTOCOL)

## Define the Measurement Layout

In [ ]:
def setupModel(data, slack: float, includeVisualAcuity: bool = True):

  assert slack >=1, "Slack must be greater than or equal to 1."

  ## Prepare for the model:

  # get results columns
  successes = data['success']
  choices = data['correctChoice']

  # define bounds
  abilityMin = {}
  abilityMax = {}

  minSuccessNav = (((data["minDistToGoal"] - data["minDistToCorrectChoice"]) * (data["minNumTurnsGoal"] - data["minNumTurnsChoice"])).min())
  minChoiceNav = ((data["minDistToCorrectChoice"] * data["minNumTurnsChoice"]).min())

  maxSuccessNav = (((data["minDistToGoal"] - data["minDistToCorrectChoice"]) * data["minNumTurnsGoal"]).max())
  maxChoiceNav = ((data["minDistToCorrectChoice"] * (data["minNumTurnsChoice"] - data["minNumTurnsChoice"])).max())

  minPermAbility = 0
  minNavAbility = min([minSuccessNav, minChoiceNav])
  if includeVisualAcuity:
    minVAcuityAbility = ((data["goalMaxDistEuclidean"]/data["mainGoalSize"]).min())

  maxPermSuccessAbility = (((data["minDistToGoal"] - data["minDistToCorrectChoice"]) * data["numChoices"]).max())
  maxPermChoiceAbility = ((data["minDistToCorrectChoice"] * data["numChoices"]).max())
  maxNavAbility = max([maxSuccessNav, maxChoiceNav])
  maxPermAbility = max([maxPermSuccessAbility, maxPermChoiceAbility])
  if includeVisualAcuity:
    maxVAcuityAbility = ((data["goalMaxDistEuclidean"]/data["mainGoalSize"]).max())

  maxReasonableMarginNav = (maxNavAbility * slack) - minNavAbility
  maxReasonableMarginOP = (maxPermAbility * slack) - minPermAbility
  if includeVisualAcuity:
    maxReasonableMarginVA = (maxVAcuityAbility * slack) - minVAcuityAbility

  abilityMin["objPermAbility"] = minPermAbility
  abilityMax["objPermAbility"] = maxPermAbility

  abilityMin["navAbility"] = minNavAbility
  abilityMax["navAbility"] = maxNavAbility

  if includeVisualAcuity:
    abilityMin["visualAcuityAbility"] = minVAcuityAbility
    abilityMax["visualAcuityAbility"] = maxVAcuityAbility

  abilityMin["lavaAbility"] = 0
  abilityMax["lavaAbility"] = 1

  abilityMin["rightAbility"] = 0
  abilityMax["rightAbility"] = 1

  abilityMin["leftAbility"] = 0
  abilityMax["leftAbility"] = 1

  abilityMin["centreAbility"] = 0
  abilityMax["centreAbility"] = 1


  ## Define the model:

  m = pm.Model()
  with m:

    # Define abilities and their priors
    objPermAbility = pm.HalfNormal("objPermAbility", sigma = maxReasonableMarginOP/2)

    navAbility = pm.HalfNormal("navAbility",sigma = maxReasonableMarginNav/2)

    if includeVisualAcuity:
      vAcuityAbility = pm.HalfNormal("visualAcuityAbility", sigma = maxReasonableMarginVA/2)

    lavaAbility = pm.HalfNormal("lavaAbility", sigma = 1)

    rightAbility = pm.HalfNormal("rightAbility", sigma = 1)

    centreAbility = pm.HalfNormal("centreAbility", sigma = 1)

    leftAbility = pm.HalfNormal("leftAbility", sigma = 1)

    # Define environment variables as Data

    goalDist = pm.Data("goalDistanceNavigation", (data["minDistToGoal"].values - data["minDistToCorrectChoice"].values))
    numChoices = pm.Data("numChoices", data["numChoices"].values)
    opTest = pm.Data("goalOccluded", data["goalBecomesAllocentricallyOccluded"].values)
    numTurnsGoal = pm.Data("navigationTurnsGoal", (data["minNumTurnsGoal"].values - data["minNumTurnsChoice"].values))
    goalSize = pm.Data("goalSize", data["mainGoalSize"].values)
    lavaPresence = pm.Data("lavaPresence", data["lavaPresence"].values)
    if includeVisualAcuity:
      goalDistanceVisual  = pm.Data("goalDistanceVisual", data["goalMaxDistEuclidean"].values)

    choiceDist = pm.Data("goalDistanceChoice", data["minDistToCorrectChoice"].values)
    numTurnsChoice = pm.Data("navigationTurnsChoice", data["minNumTurnsChoice"].values)

    goalRight = pm.Data("goalRight", data["goalRightRelToStart"].values)
    goalCentre = pm.Data("goalCentre", data["goalCentreRelToStart"].values)
    goalLeft = pm.Data("goalLeft", data["goalLeftRelToStart"].values)

    # Margins

    objPermMarginSuccess = (objPermAbility - (opTest * goalDist * numChoices))
    objPermSuccessP = pm.Deterministic("objPermSuccessP", logistic_general(objPermMarginSuccess, min = minPermAbility, max = maxPermAbility, c = 0, p = 0.99))

    objPermMarginChoice = (objPermAbility - (opTest * choiceDist * numChoices))
    objPermChoiceP = pm.Deterministic("objPermChoiceP", logistic_general(objPermMarginChoice, min = minPermAbility, max = maxPermAbility, c = 0, p = 0.99))

    lavaP = pm.Deterministic("lavaP", logistic_general((1 - ((1-lavaAbility)*lavaPresence)), min = 0, max  = 1, c = 0, p = 0.99))

    rightP = pm.Deterministic("rightP", logistic_general((1 - ((1-rightAbility)*goalRight)), min = 0, max  = 1, c = 0, p = 0.99))
    centreP = pm.Deterministic("centreP", logistic_general((1 - ((1-centreAbility)*goalCentre)), min = 0, max  = 1, c = 0, p = 0.99))
    leftP = pm.Deterministic("leftP", logistic_general((1 - ((1-leftAbility)*goalLeft)), min = 0, max  = 1, c = 0, p = 0.99))

    navSuccessP = pm.Deterministic("navSuccessP", logistic_general((navAbility - (goalDist * numTurnsGoal)), min = minNavAbility, max = maxNavAbility, c = 0, p = 0.99))
    navChoiceP = pm.Deterministic("navChoiceP", logistic_general((navAbility - (choiceDist * numTurnsChoice)), min = minNavAbility, max = maxNavAbility, c = 0, p = 0.99))

    if includeVisualAcuity:
      visualAcuityP = pm.Deterministic("visualAcuityP", logistic_general((pm.math.log(vAcuityAbility) - pm.math.log(goalDistanceVisual/goalSize)), min = np.log(minVAcuityAbility), max = np.log(maxVAcuityAbility), c = 0, p = 0.99))

    # Define final margin with non-compensatory interaction (as a product of the P values)

    if includeVisualAcuity:
      choiceP = pm.Deterministic("choiceP", navChoiceP * objPermChoiceP * visualAcuityP * rightP * centreP * leftP)
      successP = pm.Deterministic("successP", navSuccessP * objPermSuccessP * lavaP * visualAcuityP)
    else:
      choiceP = pm.Deterministic("choiceP", navChoiceP * objPermChoiceP * rightP * centreP * leftP)
      successP = pm.Deterministic("successP", navSuccessP * objPermSuccessP * lavaP)

    taskSuccess = pm.Bernoulli("taskSuccess", successP, observed=successes)
    taskChoice = pm.Bernoulli("taskChoice", choiceP, observed=choices)

  return m, abilityMin, abilityMax, successes

In [ ]:
dataset = long_data[(long_data['agent_tag_seed'].str.contains("ppo-bc_opc-strat_2023"))]
m, abilityMin, abilityMax, successes = setupModel(data=dataset, slack = 1.3)
gv = pm.model_graph.model_to_graphviz(m)
gv

## Run Complete Measurement Layouts

### Set Parameters

In [ ]:
pymc_sample_num = 2000

slackCapabilities = 1.3

chains = 4

plotFigures = False

### Test Measurement Layout

### Prepare For Training Run

In [ ]:
agent_names = ["dreamer-bc-all",
               "dreamer-bc_opc-all",
               "dreamer-bc_opc-strat",
               "dreamer-bc_opc_opt-all",
               "dreamer-bc_opc_opt-strat",
               "ppo-bc-all",
               "ppo-bc_opc-strat",
               "ppo-bc_opc-all",
               "ppo-bc_opc_opt-strat",
               "ppo-bc_opc_opt-all",
               "Random Action Agent no bias no correlation uniform step length max 20_2023",
               "Vanilla Braitenberg 15 rays over 60 degs_2023",
               "Child",
               "Child (Ablated)"
               ]

In [ ]:
results = defaultdict(list)

### Run All Models

In [ ]:
for agent in agent_names:

  results["Agent Name"].append(agent)

  rm.seed(2024)

  if agent == "Child" or agent == "Child (Ablated)":
    dataset = long_data[(long_data['agent_type_gen'].str.contains('Child'))].sort_values(by=['InstanceName']).dropna(subset = ["choiceSuccessCategorical"])
  else:
    dataset = long_data[(long_data['agent_tag_seed'].str.contains(agent))].sort_values(by=['InstanceName']).dropna(subset = ["choiceSuccessCategorical"])

  if agent == "Child (Ablated)":
    includeVA = False
    abilitiesToShow = ["objPermAbility", "navAbility", "lavaAbility", "rightAbility", "leftAbility", "centreAbility"]
  else:
    includeVA = True
    abilitiesToShow = ["objPermAbility", "visualAcuityAbility", "navAbility", "lavaAbility", "rightAbility", "leftAbility", "centreAbility"]

  train_set, test_set = train_test_split(dataset, test_size = 0.2, random_state = 2024)

  model_train, ability_min, ability_max, train_successes = setupModel(test_set,
                                                                slack = slackCapabilities,
                                                                includeVisualAcuity=includeVA,
                                                              )

  with model_train:
    data_training = pmjax.sample_numpyro_nuts(pymc_sample_num, target_accept=0.95, chains = chains, postprocessing_backend="cpu", idata_kwargs={"log_likelihood": False})

  model_train_dict = {'model' : model_train,
                      'idata' : data_training}

  # save_object(model_train_dict, f"{training_data_model_folder}/model_{agent}.pkl") #save the model just in case we need it again.

  model_test, abilityMin, abilityMax, test_successes = setupModel(test_set,
                                                                  slack = slackCapabilities,
                                                                  includeVisualAcuity=includeVA,
                                                                )

  predictionsSuccessInstance, predictionsChoiceInstance, successes, choices = predict(model_test, data_training, test_set)

  agentBrierScoreSuccess, agentCalibrationSuccess, agentRefinementSuccess = brierDecomp(predictionsSuccessInstance, successes)
  agentAggBrierScoreSuccess, agentAggCalibrationSuccess, agentAggRefinementSuccess = brierDecomp(np.repeat(np.mean(successes), len(successes)), successes)
  agentBrierScoreChoice, agentCalibrationChoice, agentRefinementChoice = brierDecomp(predictionsChoiceInstance, choices)
  agentAggBrierScoreChoice, agentAggCalibrationChoice, agentAggRefinementChoice = brierDecomp(np.repeat(np.mean(choices), len(choices)), choices)

  results["Model Brier Score Success"].append(agentBrierScoreSuccess)
  results["Model Brier Score Choice"].append(agentBrierScoreChoice)
  results["Aggregate Brier Score Success"].append(agentAggBrierScoreSuccess)
  results["Aggregate Brier Score Choice"].append(agentAggBrierScoreChoice)

  results["Model Calibration Success"].append(agentCalibrationSuccess)
  results["Model Calibration Choice"].append(agentCalibrationChoice)
  results["Aggregate Calibration Success"].append(agentAggCalibrationSuccess)
  results["Aggregate Calibration Choice"].append(agentAggCalibrationChoice)

  results["Model Refinement Success"].append(agentRefinementSuccess)
  results["Model Refinement Choice"].append(agentRefinementChoice)
  results["Aggregate Refinement Success"].append(agentAggRefinementSuccess)
  results["Aggregate Refinement Choice"].append(agentAggRefinementChoice)

  results["Model Better? (Based on Brier Score - Success)"].append(agentBrierScoreSuccess < agentAggBrierScoreSuccess)
  results["Model Better? (Based on Brier Score - Choice)"].append(agentBrierScoreChoice < agentAggBrierScoreChoice)

  results["Average Success"].append(np.mean(dataset["success"]))
  results["Average Correct Choice"].append(np.mean(dataset["correctChoice"]))

  del train_set, test_set, data_training, model_train_dict

  gc.collect()

  # now train on all data

  rm.seed(2024)

  model_all, ability_min, ability_max, successes_all = setupModel(
    data=dataset,
    slack = slackCapabilities,
    includeVisualAcuity=includeVA,
    )


  with model_all:
    data_all = pmjax.sample_numpyro_nuts(pymc_sample_num, target_accept=0.95, chains = chains, postprocessing_backend="cpu", idata_kwargs={"log_likelihood": False})

  model_test_dict = {'model' : model_all,
                      'idata' : data_all}
  save_object(model_test_dict, f"{full_data_model_folder}/model_{agent}.pkl") #save the model just in case we need it again.

  mu, sd  = analyzeAgentResults(data_all, abilitiesToShow)

  results["Object Permanence Ability Mean (All)"].append(mu["objPermAbility"])
  results["Object Permanence Ability SD (All)"].append(sd["objPermAbility"])
  results["Object Permanence Ability Min (All)"].append(ability_min["objPermAbility"])
  results["Object Permanence Ability Max (All)"].append(ability_max["objPermAbility"])

  results["Navigation Ability Mean (All)"].append(mu["navAbility"])
  results["Navigation Ability SD (All)"].append(sd["navAbility"])
  results["Navigation Ability Min (All)"].append(ability_min["navAbility"])
  results["Navigation Ability Max (All)"].append(ability_max["navAbility"])

  results["Lava Ability Mean (All)"].append(mu["lavaAbility"])
  results["Lava Ability SD (All)"].append(sd["lavaAbility"])
  results["Lava Ability Min (All)"].append(ability_min["lavaAbility"])
  results["Lava Ability Max (All)"].append(ability_max["lavaAbility"])

  results["Right Ability Mean (All)"].append(mu["rightAbility"])
  results["Right Ability SD (All)"].append(sd["rightAbility"])
  results["Right Ability Min (All)"].append(ability_min["rightAbility"])
  results["Right Ability Max (All)"].append(ability_max["rightAbility"])

  results["Left Ability Mean (All)"].append(mu["leftAbility"])
  results["Left Ability SD (All)"].append(sd["leftAbility"])
  results["Left Ability Min (All)"].append(ability_min["leftAbility"])
  results["Left Ability Max (All)"].append(ability_max["leftAbility"])

  results["Ahead Ability Mean (All)"].append(mu["centreAbility"])
  results["Ahead Ability SD (All)"].append(sd["centreAbility"])
  results["Ahead Ability Min (All)"].append(ability_min["centreAbility"])

  if includeVA:
    results["Visual Acuity Ability Mean (All)"].append(mu["visualAcuityAbility"])
    results["Visual Acuity Ability SD (All)"].append(sd["visualAcuityAbility"])
    results["Visual Acuity Ability Min (All)"].append(ability_min["visualAcuityAbility"])
    results["Visual Acuity Ability Max (All)"].append(ability_max["visualAcuityAbility"])
  else:
    results["Visual Acuity Ability Mean (All)"].append(np.nan)
    results["Visual Acuity Ability SD (All)"].append(np.nan)
    results["Visual Acuity Ability Min (All)"].append(np.nan)
    results["Visual Acuity Ability Max (All)"].append(np.nan)

  finalDF = pd.DataFrame(results)
  finalDF.to_csv(f"{csv_folder}AgentResultsAblated2.csv", index = False)

  ## figures

  if plotFigures:
    for ability in abilitiesToShow:
      trace_plot = az.plot_trace(data=data_all['posterior'][[ability]])
      plt.savefig(f"{figures_folder}/traceplot_{agent}_{ability}.png")

      forest_plot = az.plot_forest(data=data_all['posterior'][[ability]], hdi_prob=0.95, quartiles=False)
      axes_random = forest_plot.ravel()[0]
      ability_range = (abilityMax[ability] - abilityMin[ability]) * 2
      axes_random.set_xlim(left=abilityMin[ability] - ability_range, right=abilityMax[ability] + ability_range)
      plt.savefig(f"{figures_folder}/forest_{agent}_{ability}.png")

    energy_plot = az.plot_energy(data=data_all)
    plt.savefig(f"{figures_folder}/energyplot_{agent}.png")

  summary = az.summary(data_all['posterior'][abilitiesToShow], hdi_prob=0.95)

  summary.to_csv(f"{full_data_model_folder}/summary_{agent}.csv", index = False)

  print(f"Agent: {agent}")
  print(summary)

  del dataset, data_all, model_all, model_test_dict

  gc.collect()

## Plot Final Outputs

In [ ]:
abilitiesToShow = ["objPermAbility", "visualAcuityAbility", "navAbility", "lavaAbility","rightAbility", "leftAbility", "centreAbility"]

In [ ]:
for capability in abilitiesToShow:
  print(capability)
  capability_list = []
  for agent in agent_names:
    if capability != "visualAcuityAbility":
      with open(f"{full_data_model_folder}/model_{agent}.pkl", 'rb') as file:
        model_dict = pickle.load(file)
      model = model_dict['idata']['posterior'][capability]
      capability_list.append(model)
      del model, model_dict
      gc.collect()
      print(f"Posterior for {agent} on {capability} loaded.")
    else:
      if agent != "Child (Ablated)":
        with open(f"{full_data_model_folder}/model_{agent}.pkl", 'rb') as file:
          model_dict = pickle.load(file)
        model = model_dict['idata']['posterior'][capability]
        capability_list.append(model)
        del model, model_dict
        gc.collect()
        print(f"Posterior for {agent} on {capability} loaded.")
  save_object(capability_list, f"{full_data_model_folder}/{capability}_posteriors_all.pkl")
  print(f"List of posteriors for {capability} saved.")

In [ ]:
agent_name_labels = ["Dreamer 1",
                     "Dreamer 2",
                     "Dreamer 3",
                     "Dreamer 4",
                     "Dreamer 5",
                     "PPO 1",
                     "PPO 2",
                     "PPO 3",
                     "PPO 4",
                     "PPO 5",
                     "Random Action Agent",
                     "Heuristic Agent",
                     "Children",
                     "Children (Ablated)"]

colours_all = ["red",
           "red",
           "red",
           "red",
           "red",
           "chocolate",
           "chocolate",
           "chocolate",
           "chocolate",
           "chocolate",
           "lightblue",
           "palegreen",
           "deeppink",
           "deeppink"]

indexes = [0,1,2,3,4,5,6,7,8,9,10,11,12,13]

In [ ]:
abilitiesToShow = ["objPermAbility", "navAbility", "lavaAbility","rightAbility", "leftAbility", "centreAbility", "visualAcuityAbility"]

In [ ]:
for capability in abilitiesToShow:

  with open(f"{full_data_model_folder}/{capability}_posteriors_all.pkl", 'rb') as file:
        posterior_list = pickle.load(file)

  if capability == "visualAcuityAbility":
    indexes = indexes[:-1]
    colours_all = colours_all[:-1]
    agent_name_labels = agent_name_labels[:-1]

  az.plot_forest(posterior_list,
                model_names=agent_name_labels,
                var_names=capability,
                combined=True,
                hdi_prob=0.95,
                quartiles=False,
                legend=False,
                #r_hat=True,
                #ess=True,
                figsize=(10,10),
                colors = colours_all)

  plt.savefig(f"{figures_folder}/combined_forest_plot_{capability}_fullset.svg", format="svg")